In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_OUT = '/content/drive/MyDrive/lora_results'
os.makedirs(f'{DRIVE_OUT}/checkpoints', exist_ok=True)


Mounted at /content/drive


In [ ]:
!git clone https://github.com/Arnavd543/CS-4782-LoRA.git
%cd CS-4782-LoRA
%pip install -r code/requirements.txt -q

Cloning into 'CS-4782-LoRA'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 64 (delta 23), reused 51 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 25.28 KiB | 25.28 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/CS-4782-LoRA
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00


In [ ]:
import os

# Preferred: store WANDB_API_KEY in Colab Secrets and load it at runtime.
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

if not os.environ.get("WANDB_API_KEY"):
    raise ValueError("Set WANDB_API_KEY in Colab Secrets (or env vars) before training.")

In [ ]:
REPO = "/content/CS-4782-LoRA"

# First prove full fine-tuning is healthy with the Trainer-backed train.py.
DEBUG_EPOCHS = 3
FULL_FT_EPOCHS = 3

# Paper-style LoRA baseline gets more epochs because fewer parameters train.
LORA_BASELINE_EPOCHS = 10

# Lower-compute sweeps/extensions for relative comparisons.
SWEEP_EPOCHS = 3
EXT_EPOCHS = 6

## 1) Baselines
Run full fine-tuning and the paper-style LoRA baseline (Q+V, r=8, alpha=8).

In [ ]:
!python {REPO}/code/train.py --config {REPO}/code/configs/baseline.yaml --epochs {FULL_FT_EPOCHS} --gradient_accumulation_steps 1 --run_name baseline_full_ft
!python {REPO}/code/train.py --config {REPO}/code/configs/lora_r8.yaml --epochs {LORA_BASELINE_EPOCHS} --gradient_accumulation_steps 1 --run_name baseline_lora_r8_paper

# Optional Microsoft repo-style comparison (A=Kaiming, B=0, alpha=16)
# !python {REPO}/code/train.py --config {REPO}/code/configs/lora_r8.yaml --epochs {LORA_BASELINE_EPOCHS} --gradient_accumulation_steps 1 --alpha 16 --lora_init microsoft --run_name baseline_lora_r8_microsoft

In [ ]:
import json
from pathlib import Path

TARGET_ACC = 0.90
required_runs = ["baseline_full_ft", "baseline_lora_r8_paper"]
optional_runs = ["baseline_lora_r8_microsoft"]
results = {}

for run in required_runs + optional_runs:
    log_path = Path(f"{REPO}/results/logs/{run}.json")
    if not log_path.exists():
        if run in required_runs:
            print(f"Missing required log for {run}: {log_path}")
        continue
    with open(log_path, "r") as f:
        data = json.load(f)
    results[run] = float(data.get("val_accuracy", 0.0))

for run, acc in sorted(results.items()):
    status = "PASS" if acc >= TARGET_ACC else "LOW"
    print(f"{run}: {acc:.4f} [{status}]")

required_ok = all(results.get(run, 0.0) >= TARGET_ACC for run in required_runs)
if required_ok:
    print("Baseline quality check passed (>= 90% for full FT + paper-style LoRA).")
else:
    print("Baseline quality check did not pass. Increase LORA_BASELINE_EPOCHS and rerun baseline block.")

## 2) Rank sweep (lower compute)
After the baseline quality check, sweep LoRA rank with fewer epochs.

In [ ]:
for r in [1, 2, 4, 8, 16, 32]:
  !python {REPO}/code/train.py --config {REPO}/code/configs/lora_r8.yaml --epochs {SWEEP_EPOCHS} --gradient_accumulation_steps 1 --rank {r} --run_name lora_rank_{r}

## 3) Target module comparison (lower compute)
Compare query/key/value and FFN/attention-output targeting with fixed rank.

In [ ]:
modules = {
    "Wq": "query",
    "Wk": "key",
    "Wv": "value",
    "QKV": "query,key,value",
    "Attention_out": "attention.output.dense",
    "FFN_up": "intermediate.dense",
    "FFN_down": "output.dense",
}

for name, mod in modules.items():
  !python {REPO}/code/train.py --config {REPO}/code/configs/lora_r8.yaml --epochs {SWEEP_EPOCHS} --gradient_accumulation_steps 1 --rank 4 --target_modules "{mod}" --run_name lora_module_{name}

## 4) LoRA extensions (moderate compute)
Run LoRA+, LoRA+Dropout, and combined variants for comparison.

In [ ]:
!python {REPO}/code/train.py --config {REPO}/code/configs/lora_plus_r8.yaml --epochs {EXT_EPOCHS} --gradient_accumulation_steps 1 --run_name lora_plus_r8
!python {REPO}/code/train.py --config {REPO}/code/configs/lora_dropout_r8.yaml --epochs {EXT_EPOCHS} --gradient_accumulation_steps 1 --run_name lora_dropout_r8
!python {REPO}/code/train.py --config {REPO}/code/configs/lora_plus_dropout_r8.yaml --epochs {EXT_EPOCHS} --gradient_accumulation_steps 1 --run_name lora_plus_dropout_r8

## 5) Analyze and sync outputs
Generate CSV/figures and back up results + checkpoints to Google Drive.

In [ ]:
!python {REPO}/code/analyze.py

In [ ]:
!mkdir -p {DRIVE_OUT}/checkpoints
!cp -r {REPO}/results/* {DRIVE_OUT}/
!cp -r {REPO}/checkpoints/* {DRIVE_OUT}/checkpoints/

In [ ]:
import pandas as pd
from IPython.display import display

csv_paths = [
    f'{REPO}/results/baseline/baseline.csv',
    f'{REPO}/results/rank_sweep/rank_sweep.csv',
    f'{REPO}/results/module_comparison/module_comparison.csv',
    f'{REPO}/results/extensions/extensions.csv'
]

for p in csv_paths:
    try:
        df = pd.read_csv(p)
        print(f"=== {p.split('/')[-1]} ===")
        display(df)
        print()
    except Exception as e:
        print(f"Could not load {p}: {e}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path(f'{REPO}/results')

for fig_path in sorted(results_dir.rglob('*.png')):
    print(f"--- {fig_path.name} ---")
    img = mpimg.imread(str(fig_path))
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    print()